In [ ]:
'''
蜕变关系mt1构造数据, 针对存在多个triplet的text, 对其他triplet的opinion进行同义替换, 为方便后面观察inter-triplet之间是否有相互影响
'''
import json
import requests
import time
from nltk.corpus import wordnet
from gensim.models import KeyedVectors
word_vectors = KeyedVectors.load_word2vec_format('/home/karitown/experiments/aste/gensim_models/GoogleNews-vectors-negative300.bin.gz', binary=True, limit=200000)


data_path = './data/ASTE'
set_name = 'rest14'
which_split = 'test'


with open(data_path+'/'+set_name+'/'+which_split+'.txt','r',encoding='utf-8') as file:
    content =file.read()
content = [x for x in content.split('\n') if x.strip()!='']

def find_synonyms(word:str='good'):
    synonyms = []
    url = 'https://api.dictionaryapi.dev/api/v2/entries/en/'
    response = requests.get(url+word)
    if response.status_code==200:
        datas = response.json()
        for data in datas:
            if 'meanings' in data:
                meanings = data['meanings']
                for meaning in meanings:
                    for synonym in meaning['synonyms']:
                        if len(synonym.split())==1:
                            synonyms.append(synonym)
    return list(set(synonyms))

def substitute_word(text, substitute_which:str='opinion'):
    sent, labels = text.split('####')[0], eval(text.split('####')[1])
    words = sent.split(' ')
    raw2gen = {text:[]}
    mt_samples = []
    if substitute_which=='opinion':   
        if len(labels)>1:
            temps = labels.copy()
            for triplet in labels:
                samples = []
                for temp in temps:
                    if str(temp)!=str(triplet) and len(temp[1])==1 and len(set(temp[1])&set(triplet[1]))==0: # opinion不重叠

                        synonyms = find_synonyms(words[temp[1][0]])
                        synonyms = synonyms if len(synonyms)<=10 else synonyms[:10]
                        for synonym in synonyms:
                            samples.append(text.replace(words[temp[1][0]],synonym))
                            mt_samples.append(text.replace(words[temp[1][0]],synonym))
                raw2gen[text].append({str(triplet): list(set(samples))})
        return raw2gen, list(set(mt_samples))
    else:
        return []

def generate_mt1_data(origin:list, save_path:str):
    raw2gens = []
    mt_data = []
    for text in origin:
        x, y = substitute_word(text,substitute_which='opinion')
        raw2gens.append(x)
        mt_data.extend(y)
    print(len(origin), len(raw2gens), len(mt_data))
    with open(save_path+'_mt1_mapping.json','w',encoding='utf-8') as  writer1:
        writer1.write(json.dumps(raw2gens))
    with open(save_path+'_mt1.txt','w',encoding='utf-8') as  writer2:
        writer2.write('\n'.join(mt_data))

generate_mt1_data(content, data_path+'/'+set_name+'/'+which_split)

In [ ]:
'''
蜕变关系mt2构造数据, 对其他triplet进行opinion反义词替换, 为方便后面观察inter-triplet之间是否有相互影响
注意有别于针对自身triplet的opinion进行反义替换
'''
import json
import time
import requests
from nltk.corpus import wordnet

data_path = './data/ASTE'
set_name = 'laptop14'
which_split = 'test'

with open(data_path+'/'+set_name+'/'+which_split+'.txt','r',encoding='utf-8') as file:
    content = file.read()
content = [x for x in content.split('\n') if x.strip()!='']

def find_antonyms(word:str='good'):
    antonyms = []
    url = 'https://api.dictionaryapi.dev/api/v2/entries/en/'
    response = requests.get(url+word)
    if response.status_code==200:
        datas = response.json()
        for data in datas:
            if 'meanings' in data:
                meanings = data['meanings']
                for meaning in meanings:
                    for antonym in meaning['antonyms']:
                        if len(antonym.split())==1:
                            antonyms.append(antonym)
    return list(set(antonyms))

def substitute_word(text, substitute_which:str='opinion'):
    sent, labels = text.split('####')[0], eval(text.split('####')[1])
    words = sent.split(' ')
    raw2gen = {text:[]}
    mt_samples = []
    if substitute_which=='opinion':   

        if len(labels)>1:
            temps = labels.copy()
            for triplet in labels:
                samples = []
                for temp in temps:
                    if str(temp)!=str(triplet) and len(temp[1])==1 and len(set(temp[0])&set(triplet[0]))==0 and len(set(temp[1])&set(triplet[1]))==0: # aspect和opinion都不重叠

                        antonyms = find_antonyms(words[temp[1][0]])
                        antonyms = antonyms if len(antonyms)<=10 else antonyms[:10]
                        for antonym in antonyms:
                            samples.append(text.replace(words[temp[1][0]],antonym))
                            mt_samples.append(text.replace(words[temp[1][0]],antonym))
                raw2gen[text].append({str(triplet): list(set(samples))})

        return raw2gen, list(set(mt_samples))
    else:
        return []

def generate_mt2_data(origin:list, save_path:str):
    raw2gens = []
    mt_data = []
    for text in origin:
        x, y = substitute_word(text,substitute_which='opinion')
        raw2gens.append(x)
        mt_data.extend(y)
    print(len(origin), len(raw2gens), len(mt_data))
    with open(save_path+'_mt2_mapping.json','w',encoding='utf-8') as  writer1:
        writer1.write(json.dumps(raw2gens))
    with open(save_path+'_mt2.txt','w',encoding='utf-8') as  writer2:
        writer2.write('\n'.join(mt_data))

generate_mt2_data(content, data_path+'/'+set_name+'/'+which_split) 

In [ ]:
'''
蜕变关系mt3构造数据, add一些相反情感的triplet, 以合适的句式添加进去, 为方便后面观察inter-triplet之间是否有相互影响
'''
import json
import random
from parrot import Parrot
import torch
import warnings

data_path = './data/ASTE'
set_name = 'laptop14'
which_split = 'test'
with open(data_path+'/'+set_name+'/'+which_split+'.txt','r',encoding='utf-8') as file:
    content = [x for x in file.read().split('\n') if x.strip()!='']

def get_sublist(main_list,index_list):
    result = []
    for index in index_list:
        result.append(main_list[index])
    return result

def random_state(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
       torch.cuda.manual_seed_all(seed)
random_state(1234)

def get_subtext_from_ao(ao_list:list)->dict:
    _parrot = Parrot(model_tag='/data1/home/tangsp/hf_models/parrot_paraphraser_on_T5')
    ao_subtext_dict = dict()
    for ao in ao_list:
        aspect, opinion = ao.split('####')
        para_phrases = _parrot.augment(input_phrase=aspect+' is '+opinion,
                                      use_gpu=False,
                                      diversity_ranker="levenshtein",
                                      do_diverse=True,
                                      max_return_phrases = 10,
                                      max_length=32,
                                      adequacy_threshold = 0.50,
                                      fluency_threshold = 0.50)
        if para_phrases:
            for para_phrase, _ in para_phrases:
                if len(para_phrase) >= len(opinion+' '+aspect): 
                    if aspect+'####'+opinion not in ao_subtext_dict:
                        ao_subtext_dict[aspect+'####'+opinion] = [para_phrase]
                    else:
                        ao_subtext_dict[aspect+'####'+opinion].append(para_phrase)
        
    return ao_subtext_dict

def get_pos_neg_aos():
    sets = ['laptop14', 'rest14', 'rest15', 'rest16']
    content = []
    for _set in sets:
        with open('./data/ASTE/'+_set+'/test.txt','r',encoding='utf-8') as file:
            content.extend([x for x in file.read().split('\n') if x.strip()!=''])
    content = list(set(content))
    print(len(content))

    pos_ao_list = []
    neg_ao_list = []
    for text in content:
        words, labels = text.split('####')[0].split(' '), eval(text.split('####')[1])
        for label in labels:
            if label[2] == 'POS':
                pos_ao_list.append(' '.join(get_sublist(words,label[0]))+'####'+' '.join(get_sublist(words,label[1]))) 
            elif label[2] == 'NEG':
                neg_ao_list.append(' '.join(get_sublist(words,label[0]))+'####'+' '.join(get_sublist(words,label[1])))
    pos_list = list(set(pos_ao_list))
    neg_list = list(set(neg_ao_list))

    pos_ao_paraphrase_dict = get_subtext_from_ao(pos_list)
    neg_ao_paraphrase_dict = get_subtext_from_ao(neg_list)

    with open('./data/paraphrase_data/pos_ao_paraphrase.json', 'w') as pos_file:
        json.dump(pos_ao_paraphrase_dict, pos_file)

    with open('./data/paraphrase_data/neg_ao_paraphrase.json', 'w') as neg_file:
        json.dump(neg_ao_paraphrase_dict, neg_file)

def add_triplet(pos_ao_paraphrase_dict, neg_ao_paraphrase_dict, text, substitute_which:str='opinion'):  
    pos_keys = list(pos_ao_paraphrase_dict.keys())
    neg_keys = list(neg_ao_paraphrase_dict.keys())

    sent, labels, labels_list = text.split('####')[0], text.split('####')[1], eval(text.split('####')[1])
    words = sent.split(' ')
    raw2gen = {text:[]}
    mt_samples = []
    if substitute_which=='opinion':   
        for label in labels_list:  
            samples =[]
            if label[2]=='POS':
                random.shuffle(pos_keys)
                for neg_key in neg_keys:
                    if neg_key.split('####')[0] not in ' '.join(get_sublist(words, label[0])): 
                        new_sent = sent + ' ' + random.choice(neg_ao_paraphrase_dict[neg_key])
                        new_text = new_sent+'####'+labels
                        samples.append(new_text)
                        mt_samples.append(new_text)
                    if len(samples)==10:
                        break
            elif label[2]=='NEG':
                random.shuffle(pos_keys)
                for pos_key in pos_keys:
                    if pos_key.split('####')[0] not in ' '.join(get_sublist(words, label[0])): 
                        new_sent = sent + ' ' + random.choice(pos_ao_paraphrase_dict[pos_key])
                        new_text = new_sent+'####'+labels
                        samples.append(new_text)
                        mt_samples.append(new_text)
                    if len(samples)==10:
                        break
            raw2gen[text].append({str(label): samples})
        return raw2gen, list(set(mt_samples))
    else:
        return []

def generate_mt3_data(origin:list, save_path:str):
    with open('./data/paraphrase_data/pos_ao_paraphrase.json', 'r') as pos_file:
        pos_ao_paraphrase_dict = json.load(pos_file)
    with open('./data/paraphrase_data/neg_ao_paraphrase.json', 'r') as neg_file:
        neg_ao_paraphrase_dict = json.load(neg_file)
    raw2gens = []
    mt_data = []
    for text in origin:
        x, y = add_triplet(pos_ao_paraphrase_dict, neg_ao_paraphrase_dict, text,substitute_which='opinion')
        raw2gens.append(x)
        mt_data.extend(y)
    print(len(origin), len(raw2gens), len(mt_data))
    with open(save_path+'_mt3_mapping.json','w',encoding='utf-8') as  writer1:
        writer1.write(json.dumps(raw2gens))
    with open(save_path+'_mt3.txt','w',encoding='utf-8') as  writer2:
        writer2.write('\n'.join(mt_data))
   
generate_mt3_data(content, save_path=data_path+'/'+set_name+'/'+which_split)

In [ ]:
'''
蜕变关系mt4构造数据, 对于有多个triplet的text, 针对某个triplet, mask掉其他无关的triplet, 为方便后面观察inter-triplet之间是否有相互影响
'''
import json

data_path = './data/ASTE'
set_name = 'laptop14'
which_split = 'test'
with open(data_path+'/'+set_name+'/'+which_split+'.txt','r',encoding='utf-8') as file:
    content = [x for x in file.read().split('\n') if x.strip()!='' and len(eval(x.split('####')[1]))>1]

def mask_other(text):
    sent, labels, labels_list = text.split('####')[0], text.split('####')[1], eval(text.split('####')[1])
    words = sent.split(' ')
    raw2gen = {text:[]}
    mt_samples = []
   
    ao_triplet_dict = dict()   
    for label in labels_list:   
        ao_triplet_dict[str(label[0])+'####'+str(label[1])] = [] 
    for label in labels_list:  
        for key in ao_triplet_dict.keys():
            if key!=str(label[0])+'####'+str(label[1]):
                ao_triplet_dict[key].append(label)

    for ao, other_triplet in ao_triplet_dict.items():
        aspect, opinion = eval(ao.split('####')[0]), eval(ao.split('####')[1])
        samples = []
        for triplet in other_triplet:  
            _words = words.copy() 
            if len(set(aspect)&set(triplet[0]))!=0: 
                for index in triplet[1]:
                    _words[index] = '[UNK]'
                new_text = ' '.join(_words)+'####'+labels

                samples.append(new_text)
                mt_samples.append(new_text)
            elif len(set(opinion)&set(triplet[1]))!=0:  
                for index in triplet[0]:
                    _words[index] = '[UNK]'
                new_text = ' '.join(_words)+'####'+labels

                samples.append(new_text)
                mt_samples.append(new_text)
            else:   
                for index in triplet[0]:
                    _words[index] = '[UNK]'
                for index in triplet[1]:
                    _words[index] = '[UNK]'
                new_text = ' '.join(_words)+'####'+labels

                samples.append(new_text)
                mt_samples.append(new_text)
        for label in labels_list:
            if label[0]==aspect and label[1]==opinion:
                raw2gen[text].append({str(label): samples})
    return raw2gen, list(set(mt_samples))

def generate_mt4_data(origin, save_path):
    raw2gens = []
    mt_data = []
    for text in origin:
        x, y = mask_other(text)
        raw2gens.append(x)
        mt_data.extend(y)
    print(len(origin), len(raw2gens), len(mt_data))
    with open(save_path+'_mt4_mapping.json','w',encoding='utf-8') as  writer1:
        writer1.write(json.dumps(raw2gens))
    with open(save_path+'_mt4.txt','w',encoding='utf-8') as  writer2:
        writer2.write('\n'.join(mt_data))

generate_mt4_data(content, save_path=data_path+'/'+set_name+'/'+which_split)

In [ ]:
'''
蜕变关系mt5构造数据, 仅对aspect作同义词或上位词替换, 为方便后面观察intra-triplet的影响
'''
import json
import requests
from nltk.corpus import wordnet
from gensim.models import KeyedVectors
word_vectors = KeyedVectors.load_word2vec_format('../../word2vec_models/GoogleNews-vectors-negative300.bin.gz', binary=True, limit=200000)

data_path = './data/ASTE'
set_name = 'laptop14'
which_split = 'test'
with open(data_path+'/'+set_name+'/'+which_split+'.txt','r',encoding='utf-8') as file:
    content = [x for x in file.read().split('\n') if x.strip()!='' and len(eval(x.split('####')[1]))>1]
print(len(content))

def find_synonyms(word):
    synonyms = []
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            if lemma.name().lower()!= word.lower() and '_' not in lemma.name().lower() and len(lemma.name().lower())>1:
                synonyms.append(lemma.name().lower())
    return list(set(synonyms))

def find_hypernyms(word):
    hypernyms = []
    synsets = wordnet.synsets(word)  
    if synsets:
        for synset in synsets:
            _hypernyms = synset.hypernyms()  
            for h in _hypernyms:
                if '_' not in h.name().split('.')[0] and len(h.name().split('.')[0])>1 and h.name().split('.')[0]!= word.lower():
                    hypernyms.append(h.name().lower().split('.')[0])
    return list(set(hypernyms))

def substitute_word(text, substitute_which:str='aspect'):
    sent, labels = text.split('####')[0], eval(text.split('####')[1])
    words = sent.split(' ')
    raw2gen = {text:[]}
    mt_samples = []
    if substitute_which=='aspect':   
        for label in labels:
            samples = []
            if len(label[0])==1:
                hypernyms = find_hypernyms(words[label[0][0]])
                nyms = find_synonyms(words[label[0][0]]) + hypernyms
                nyms = sorted(nyms, key=len, reverse=True)
                nyms = nyms if len(nyms)<=10 else nyms[:10]
                for synonym in list(set(nyms)):
                    temp_words = words.copy()
                    temp_words[label[0][0]] = synonym
                    samples.append(' '.join(temp_words)+'####'+text.split('####')[1])
                    mt_samples.append(' '.join(temp_words)+'####'+text.split('####')[1])
                raw2gen[text].append({str(label): list(set(samples))})
        return raw2gen, list(set(mt_samples))
    else:
        return []

def generate_mt5_data(origin:list, save_path:str):
    raw2gens = []
    mt_data = []
    for text in origin:
        x, y = substitute_word(text,substitute_which='aspect')
        raw2gens.append(x)
        mt_data.extend(y)
    print(len(origin), len(raw2gens), len(mt_data))
    with open(save_path+'_mt5_mapping.json','w',encoding='utf-8') as  writer1:
        writer1.write(json.dumps(raw2gens))
    with open(save_path+'_mt5.txt','w',encoding='utf-8') as  writer2:
        writer2.write('\n'.join(mt_data))

generate_mt5_data(content, data_path+'/'+set_name+'/'+which_split) 


In [ ]:
'''
蜕变关系mt6构造数据, 仅对opinion作同义词替换, 为方便后面观察intra-triplet的影响
'''
import re
import json
import requests
from nltk.corpus import wordnet
from gensim.models import KeyedVectors
word_vectors = KeyedVectors.load_word2vec_format('../../word2vec_models/GoogleNews-vectors-negative300.bin.gz', binary=True, limit=200000)

data_path = './data/ASTE'
set_name = 'laptop14'
which_split = 'test'
with open(data_path+'/'+set_name+'/'+which_split+'.txt','r',encoding='utf-8') as file:
    content = [x for x in file.read().split('\n') if x.strip()!='' and len(eval(x.split('####')[1]))>1]
print(len(content))

def is_all_punctuation(text):
    result = re.findall(r'[A-Za-z]+', text)
    if len(result)==0:
        return True
    else:
        return False

def find_synonyms(word:str='good'):
    synonyms = []
    url = 'https://api.dictionaryapi.dev/api/v2/entries/en/'
    response = requests.get(url+word)
    if response.status_code==200:
        datas = response.json()
        for data in datas:
            if 'meanings' in data:
                meanings = data['meanings']
                for meaning in meanings:
                    for synonym in meaning['synonyms']:
                        if len(synonym.split())==1 and not is_all_punctuation(synonym):
                            synonyms.append(synonym)
    return list(set(synonyms))

def substitute_word(text, substitute_which:str='aspect'):
    sent, labels = text.split('####')[0], eval(text.split('####')[1])
    words = sent.split(' ')
    raw2gen = {text:[]}
    mt_samples = []
    if substitute_which=='opinion': 
        for label in labels:
            samples = []
            if len(label[1])==1:
                sysnonyms = find_synonyms(words[label[1][0]])
                sysnonyms = sysnonyms if len(sysnonyms)<=10 else sysnonyms[:10]
                for synonym in list(set(sysnonyms)):
                    samples.append(text.replace(words[label[1][0]],synonym))
                    mt_samples.append(text.replace(words[label[1][0]],synonym))
                raw2gen[text].append({str(label): list(set(samples))})

        return raw2gen, list(set(mt_samples))
    else:
        return []

def generate_mt6_data(origin:list, save_path:str):
    raw2gens = []
    mt_data = []
    for text in origin:
        x, y = substitute_word(text,substitute_which='opinion')
        raw2gens.append(x)
        mt_data.extend(y)
    print(len(origin), len(raw2gens), len(mt_data))
    with open(save_path+'_mt6_mapping.json','w',encoding='utf-8') as  writer1:
        writer1.write(json.dumps(raw2gens))
    with open(save_path+'_mt6.txt','w',encoding='utf-8') as  writer2:
        writer2.write('\n'.join(mt_data))

generate_mt6_data(content, data_path+'/'+set_name+'/'+which_split)

In [ ]:
'''
蜕变关系mt7构造数据, 对aspect和opinion作typos变化, 为方便后面观察intra-triplet的影响
'''
import re
import json
import random

data_path = './data/ASTE'
set_name = 'laptop14'
which_split = 'test'
substitute_which = 'aspect&opinion' 
sub_type = 'shuffle'   

with open(data_path+'/'+set_name+'/'+which_split+'.txt','r',encoding='utf-8') as file:
    content = [x for x in file.read().split('\n') if x.strip()!='' and len(eval(x.split('####')[1]))>1]
print(len(content))

def is_all_punctuation(text):
    result = re.findall(r'[A-Za-z]+', text)
    if len(result)==0:
        return True
    else:
        return False
    
def generate_typo_replace(substr):
    keyboard_layout = {
        'q': 'asw',
        'w': 'qasde',
        'e': 'wsdfr',
        'r': 'edfgt',
        't': 'rfghy',
        'y': 'tghju',
        'u': 'yhjki',
        'i': 'ujklo',
        'o': 'iklp',
        'p': 'ol',
        'a': 'zxswq',
        's': 'qazxcdew',
        'd': 'wsxcvfre',
        'f': 'edcvbgtr',
        'g': 'rfvbnhyt',
        'h': 'tgbnmjuy',
        'j': 'yhnmkiu',
        'k': 'ujmloi',
        'l': 'iko',
        'z': 'xsa',
        'x': 'zcdsa',
        'c': 'xvfds',
        'v': 'cbgfd',
        'b': 'vnhgf',
        'n': 'bmjhg',
        'm': 'nkjh'
    }
    chars = list(substr.lower())
    isOk = False
    while not isOk:
        index = random.choice(range(len(chars)))

        if chars[index] in keyboard_layout:
            chars[index] = random.choice(keyboard_layout[chars[index]])
            isOk = True
    return ''.join(chars)

def generate_typo_swap(substr): 
    words = substr.split(' ')

    swap_word = max(words,key=len)
    chars = list(swap_word)
    isOk = False
    while not isOk:
        if len(swap_word)>2:
            indices = random.sample(range(len(swap_word)),2)
            if chars[indices[0]]!=chars[indices[1]]:
                chars[indices[0]], chars[indices[1]] = chars[indices[1]], chars[indices[0]]
                isOk = True
        else:
            isOk = True
    return substr.replace(swap_word,''.join(chars))

def generate_typo_shuffle(substr):
    words = substr.split(' ')

    if len(substr)>len(words):  
        isOk = False
        new_substr = ''
        while not isOk:
            shuffle_word = random.choice(words)
            chars = list(shuffle_word)
            if len(shuffle_word)>1:
                random.shuffle(chars)
                new_substr = substr.replace(shuffle_word,''.join(chars))
                isOk=True
        return new_substr
    else:
        return substr
    
def substitute_word(text, substitute_which:str='aspect&opinion'):
    sent, labels = text.split('####')[0], eval(text.split('####')[1])
    words = sent.split(' ')
    raw2gen = {text:[]}
    mt_samples = []
    if substitute_which=='aspect&opinion':
        for label in labels:
            samples = []

            replace_aspect = ' '.join([words[i] for i in label[0]])
            replace_opinion = ' '.join([words[i] for i in label[1]])
            if not is_all_punctuation(replace_aspect) and not is_all_punctuation(replace_opinion):
                new_text = sent.replace(replace_aspect, generate_typo_replace(replace_aspect)).replace(replace_opinion, generate_typo_replace(replace_opinion))+'####'+text.split('####')[1]
                samples.append(new_text)
                mt_samples.append(new_text)
            
            raw2gen[text].append({str(label): list(set(samples))})

        return raw2gen, list(set(mt_samples))
    elif substitute_which=='aspect':
        for label in labels:
            samples = []

            replace_aspect = ' '.join([words[i] for i in label[0]])
            if not is_all_punctuation(replace_aspect):
                new_text = sent.replace(replace_aspect, generate_typo_replace(replace_aspect))+'####'+text.split('####')[1]
                samples.append(new_text)
                mt_samples.append(new_text)

            swap_aspect = ' '.join([words[i] for i in label[0]])
            if not is_all_punctuation(swap_aspect):
                new_text = sent.replace(swap_aspect, generate_typo_swap(swap_aspect))+'####'+text.split('####')[1]
                samples.append(new_text)
                mt_samples.append(new_text) 
            
            shuffle_aspect = ' '.join([words[i] for i in label[0]])
            if not is_all_punctuation(shuffle_aspect):
                new_text = sent.replace(shuffle_aspect, generate_typo_shuffle(shuffle_aspect))+'####'+text.split('####')[1]
                samples.append(new_text)
                mt_samples.append(new_text)

            raw2gen[text].append({str(label): list(set(samples))})

        return raw2gen, list(set(mt_samples))
    elif substitute_which=='opinion':
        for label in labels:
            samples = []

            replace_opinion = ' '.join([words[i] for i in label[1]])
            if not is_all_punctuation(replace_opinion):
                new_text = sent.replace(replace_opinion, generate_typo_replace(replace_opinion))+'####'+text.split('####')[1]
                samples.append(new_text)
                mt_samples.append(new_text)

            swap_opinion = ' '.join([words[i] for i in label[1]])
            if not is_all_punctuation(swap_opinion):
                new_text = sent.replace(swap_opinion, generate_typo_swap(swap_opinion))+'####'+text.split('####')[1]
                samples.append(new_text)
                mt_samples.append(new_text)
            
            shuffle_opinion = ' '.join([words[i] for i in label[1]])
            if not is_all_punctuation(shuffle_opinion):
                new_text = sent.replace(shuffle_opinion, generate_typo_shuffle(shuffle_opinion))+'####'+text.split('####')[1]
                samples.append(new_text)
                mt_samples.append(new_text)
            
            raw2gen[text].append({str(label): list(set(samples))})

        return raw2gen, list(set(mt_samples))
    
    else:
        return []

def generate_mt7_data(origin:list, save_path:str):
    raw2gens = []
    mt_data = []
    for text in origin:
        x, y = substitute_word(text,substitute_which=substitute_which)  
        raw2gens.append(x)
        mt_data.extend(y)
    print(len(origin), len(raw2gens), len(mt_data))
    if substitute_which=='aspect&opinion':
        if sub_type == 'all':
            with open(f'{save_path}_mt7_mapping.json','w',encoding='utf-8') as  writer1:
                writer1.write(json.dumps(raw2gens))
            with open(f'{save_path}_mt7.txt','w',encoding='utf-8') as  writer2:
                writer2.write('\n'.join(mt_data))
        else:
            with open(f'{save_path}_mt7-{sub_type}_mapping.json','w',encoding='utf-8') as  writer1:
                writer1.write(json.dumps(raw2gens))
            with open(f'{save_path}_mt7-{sub_type}.txt','w',encoding='utf-8') as  writer2:
                writer2.write('\n'.join(mt_data))
    else:
        with open(f'{save_path}_mt7-{substitute_which}_mapping.json','w',encoding='utf-8') as  writer1:
            writer1.write(json.dumps(raw2gens))
        with open(f'{save_path}_mt7-{substitute_which}.txt','w',encoding='utf-8') as  writer2:
            writer2.write('\n'.join(mt_data))

generate_mt7_data(content, data_path+'/'+set_name+'/'+which_split) 

In [ ]:
'''
蜕变关系mt8构造数据, 同时对aspect和opinion作同义词替换, 为方便后面观察intra-triplet的影响
'''
import re
import json
import time
import requests
from nltk.corpus import wordnet
from gensim.models import KeyedVectors
word_vectors = KeyedVectors.load_word2vec_format('../../word2vec_models/GoogleNews-vectors-negative300.bin.gz', binary=True, limit=200000)

data_path = './data/ASTE'
set_name = 'laptop14'
which_split = 'test'
with open(data_path+'/'+set_name+'/'+which_split+'.txt','r',encoding='utf-8') as file:
    content = [x for x in file.read().split('\n') if x.strip()!='' and len(eval(x.split('####')[1]))>1]
print(len(content))

def is_all_punctuation(text):
    result = re.findall(r'[A-Za-z]+', text)
    if len(result)==0:
        return True
    else:
        return False
    
def find_aspect_synonyms(word):
    synonyms = []
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            if lemma.name().lower()!= word.lower() and '_' not in lemma.name().lower() and len(lemma.name().lower())>1:
                synonyms.append(lemma.name().lower())
    return list(set(synonyms))
def find_aspect_hypernyms(word):
    hypernyms = []
    synsets = wordnet.synsets(word)  
    if synsets:
        for synset in synsets:
            _hypernyms = synset.hypernyms()  
            for h in _hypernyms:
                if '_' not in h.name().split('.')[0] and len(h.name().split('.')[0])>1 and h.name().split('.')[0]!= word.lower():
                    hypernyms.append(h.name().lower().split('.')[0])

    return list(set(hypernyms))

def find_opinion_synonyms(word:str='good'):
    synonyms = []
    url = 'https://api.dictionaryapi.dev/api/v2/entries/en/'
    response = requests.get(url+word)
    if response.status_code==200:
        datas = response.json()
        for data in datas:
            if 'meanings' in data:
                meanings = data['meanings']
                for meaning in meanings:
                    for synonym in meaning['synonyms']:
                        if len(synonym.split())==1 and not is_all_punctuation(synonym):
                            synonyms.append(synonym)
    return list(set(synonyms))

def substitute_word(text, substitute_which:str='aspect'):
    sent, labels = text.split('####')[0], eval(text.split('####')[1])
    words = sent.split(' ')
    raw2gen = {text:[]}
    mt_samples = []
    if substitute_which=='opinion': 
        for label in labels:
            samples = []
            if len(label[0])==1 and len(label[1])==1:   

                hypernyms = find_aspect_hypernyms(words[label[0][0]])
                nyms = find_aspect_synonyms(words[label[0][0]]) + hypernyms
                nyms = sorted(nyms, key=len, reverse=True)
                nyms = nyms if len(nyms)<=10 else nyms[:10]

                sysnonyms = find_opinion_synonyms(words[label[1][0]])
                time.sleep(1)
                sysnonyms = sysnonyms if len(sysnonyms)<=10 else sysnonyms[:10]

                pair_num = len(nyms) if len(nyms)<=len(sysnonyms) else len(sysnonyms)
                for i in range(pair_num):
                    new_text = sent.replace(words[label[0][0]], nyms[i]).replace(words[label[1][0]], sysnonyms[i])+'####'+text.split('####')[1]
                    samples.append(new_text)
                    mt_samples.append(new_text)
                raw2gen[text].append({str(label): list(set(samples))})
        return raw2gen, list(set(mt_samples))
    else:
        return []

def generate_mt8_data(origin:list, save_path:str):
    raw2gens = []
    mt_data = []
    for text in origin:
        x, y = substitute_word(text,substitute_which='opinion')
        raw2gens.append(x)
        mt_data.extend(y)
    print(len(origin), len(raw2gens), len(mt_data))
    with open(save_path+'_mt8_mapping.json','w',encoding='utf-8') as  writer1:
        writer1.write(json.dumps(raw2gens))
    with open(save_path+'_mt8.txt','w',encoding='utf-8') as  writer2:
        writer2.write('\n'.join(mt_data))

generate_mt8_data(content, data_path+'/'+set_name+'/'+which_split) 

In [ ]:
'''
蜕变关系mt9构造数据, 仅对opinion作反义词替换, 为方便后面观察intra-triplet的影响
'''
import re
import json
from nltk.corpus import wordnet

data_path = './data/ASTE'
set_name = 'laptop14'
which_split = 'test'
with open(data_path+'/'+set_name+'/'+which_split+'.txt','r',encoding='utf-8') as file:
    content = [x for x in file.read().split('\n') if x.strip()!='' and len(eval(x.split('####')[1]))>1]
print(len(content))

def is_all_punctuation(text):
    result = re.findall(r'[A-Za-z]+', text)
    if len(result)==0:
        return True
    else:
        return False

def find_antonyms(word):
    antonyms = []
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            for antonym in lemma.antonyms():
                antonyms.append(antonym.name())
    return list(set(antonyms))

def substitute_word(text, substitute_which:str='aspect'):
    sent, labels = text.split('####')[0], eval(text.split('####')[1])
    words = sent.split(' ')
    raw2gen = {text:[]}
    mt_samples = []
    if substitute_which=='opinion':
        for label in labels:
            samples = []
            if len(label[1])==1 and label[2]!='NEU':
                antonyms = find_antonyms(words[label[1][0]])
                antonyms = antonyms if len(antonyms)<=10 else antonyms[:10]
                for antonym in list(set(antonyms)):
                    samples.append(text.replace(words[label[1][0]],antonym))
                    mt_samples.append(text.replace(words[label[1][0]],antonym))
                if label[2]=='POS':
                    raw2gen[text].append({str(label).replace('POS','NEG'): list(set(samples))})
                elif label[2]=='NEG':
                    raw2gen[text].append({str(label).replace('NEG','POS'): list(set(samples))})

        return raw2gen, list(set(mt_samples))
    else:
        return []

def generate_mt9_data(origin:list, save_path:str):
    raw2gens = []
    mt_data = []
    for text in origin:
        x, y = substitute_word(text,substitute_which='opinion')
        raw2gens.append(x)
        mt_data.extend(y)
    print(len(origin), len(raw2gens), len(mt_data))
    with open(save_path+'_mt9_mapping.json','w',encoding='utf-8') as  writer1:
        writer1.write(json.dumps(raw2gens))
    with open(save_path+'_mt9.txt','w',encoding='utf-8') as  writer2:
        writer2.write('\n'.join(mt_data))

generate_mt9_data(content, data_path+'/'+set_name+'/'+which_split) 